In [2]:
import pandas as pd
df = pd.DataFrame({
    "门店": ["A","A","A","B","B","B"],
    "品类": ["水果","蔬菜","水果","水果","蔬菜","蔬菜"],
    "销售额": [100, 200, 150, 120, 210, 240],
    "成本":  [60,  130,  90,  70,  140,  150],
})
g = df.groupby("门店")
print(g["销售额"].sum())
print(g[["销售额","成本"]].mean())
print(g.size())
print(g["销售额"].agg(["sum","mean","count"]))
print(df.groupby(["门店","品类"])["销售额"].sum())
print(df.groupby(["门店","品类"])["销售额"].sum().unstack(fill_value=0))

门店
A    450
B    570
Name: 销售额, dtype: int64
      销售额          成本
门店                   
A   150.0   93.333333
B   190.0  120.000000
门店
A    3
B    3
dtype: int64
    sum   mean  count
门店                   
A   450  150.0      3
B   570  190.0      3
门店  品类
A   水果    250
    蔬菜    200
B   水果    120
    蔬菜    450
Name: 销售额, dtype: int64
品类   水果   蔬菜
门店          
A   250  200
B   120  450


In [ ]:
stat = df.groupby("门店").agg(
    总销售额 = ("销售额","sum"),
    平均销售额=("销售额","mean"),
    订单数=("销售额","count"),
    毛利率=("成本",lambda x:1 - x.sum()/df.loc[x.index,"销售额"].sum()),
).reset_index()
print(stat)
df["门店总额"] = df.groupby("门店")["销售额"].transform("sum")
df["占比"] = df["销售额"] / df["门店总额"]
df["去均值"] = df["销售额"] - df.groupby("品类")["销售额"].transform("mean")
print(df.groupby("门店").filter(lambda g:g["销售额"].sum() > 500))
print(df.groupby("品类")["销售额"].value_counts(bins=4))

  门店  总销售额  平均销售额  订单数       毛利率
0  A   450  150.0    3  0.377778
1  B   570  190.0    3  0.368421
  门店  品类  销售额   成本  门店总额        占比        去均值
3  B  水果  120   70   570  0.210526  -3.333333
4  B  蔬菜  210  140   570  0.368421  -6.666667
5  B  蔬菜  240  150   570  0.421053  23.333333
品类  销售额             
水果  (99.949, 112.5]     1
    (112.5, 125.0]      1
    (137.5, 150.0]      1
    (125.0, 137.5]      0
蔬菜  (199.959, 210.0]    2
    (230.0, 240.0]      1
    (210.0, 220.0]      0
    (220.0, 230.0]      0
Name: count, dtype: int64


In [10]:
orders = pd.DataFrame({
    "订单号": [1, 2, 3, 4], "用户ID": [101, 102, 101, 103], "金额": [100, 200, 150, 300],
})
users = pd.DataFrame({
    "用户ID": [101, 102, 104], "城市": ["北京", "上海", "广州"], "等级": ["A", "B", "C"],
})
print(pd.merge(orders,users,on="用户ID",how="inner"))
print(pd.merge(orders,users,on="用户ID",how="left"))
print(pd.merge(orders,users,on="用户ID",how="outer"))
print(pd.merge(orders,users,how="left",indicator=True).query("_merge == 'left_only'"))


   订单号  用户ID   金额  城市 等级
0    1   101  100  北京  A
1    2   102  200  上海  B
2    3   101  150  北京  A
   订单号  用户ID   金额   城市   等级
0    1   101  100   北京    A
1    2   102  200   上海    B
2    3   101  150   北京    A
3    4   103  300  NaN  NaN
   订单号  用户ID     金额   城市   等级
0  1.0   101  100.0   北京    A
1  3.0   101  150.0   北京    A
2  2.0   102  200.0   上海    B
3  4.0   103  300.0  NaN  NaN
4  NaN   104    NaN   广州    C
   订单号  用户ID   金额   城市   等级     _merge
3    4   103  300  NaN  NaN  left_only


In [13]:
df1 = pd.DataFrame({"a":[1,2]});df2 = pd.DataFrame({"a":[3,4]})
print(pd.concat([df1,df2],ignore_index=True))
print(pd.concat([df1,df2],axis=1))
print(orders.set_index("用户ID").join(users.set_index("用户ID"),how="left"))
merged = pd.merge(orders,users,on="用户ID",how="left")
assert len(merged) == len(orders),"左连接行数膨胀，检查是否一对多！"
print(merged.isna().sum())

   a
0  1
1  2
2  3
3  4
   a  a
0  1  3
1  2  4
      订单号   金额   城市   等级
用户ID                    
101     1  100   北京    A
102     2  200   上海    B
101     3  150   北京    A
103     4  300  NaN  NaN
订单号     0
用户ID    0
金额      0
城市      1
等级      1
dtype: int64


In [15]:
df = pd.DataFrame({
    "日期": ["2024-01-01", "2024-01-02", "2024-01-03", "2024-02-01"],
    "销售额": [100, 150, 130, 200],
})
df["日期"] = pd.to_datetime(df["日期"])
df = df.set_index("日期")
s = pd.Series(pd.date_range("2024-01-01",periods=5,freq="D"))
print(s.dt.year,s.dt.month,s.dt.day_name(),s.dt.is_month_end)
print(s.dt.strftime("%Y年%m月"))
daily = df.resample("D").asfreq()
daily = df.resample("D").ffill()
print(df.resample("ME")["销售额"].sum())
print(df.resample("W")["销售额"].agg(["sum","mean"]))

0    2024
1    2024
2    2024
3    2024
4    2024
dtype: int32 0    1
1    1
2    1
3    1
4    1
dtype: int32 0       Monday
1      Tuesday
2    Wednesday
3     Thursday
4       Friday
dtype: str 0    False
1    False
2    False
3    False
4    False
dtype: bool
0    2024年01月
1    2024年01月
2    2024年01月
3    2024年01月
4    2024年01月
dtype: str
日期
2024-01-31    380
2024-02-29    200
Freq: ME, Name: 销售额, dtype: int64
            sum        mean
日期                         
2024-01-07  380  126.666667
2024-01-14    0         NaN
2024-01-21    0         NaN
2024-01-28    0         NaN
2024-02-04  200  200.000000


In [22]:
print(df["销售额"].rolling(3).mean())
print(df["销售额"].rolling(3,min_periods=1).sum())
print(df["销售额"].shift(1))
df["环比"] = df["销售额"].pct_change()
print(df.expanding().mean())
print((pd.Timestamp("2024-03-01") - pd.Timestamp("2024-01-01")).days)
print(pd.date_range("2024-01-01","2024-01-10",freq="2D"))
ts = pd.Timestamp("2024-01-01",tz="Asia/Shanghai")
print(ts.tz_convert("UTC"))

日期
2024-01-01           NaN
2024-01-02           NaN
2024-01-03    126.666667
2024-02-01    160.000000
Name: 销售额, dtype: float64
日期
2024-01-01    100.0
2024-01-02    250.0
2024-01-03    380.0
2024-02-01    480.0
Name: 销售额, dtype: float64
日期
2024-01-01      NaN
2024-01-02    100.0
2024-01-03    150.0
2024-02-01    130.0
Name: 销售额, dtype: float64
                   销售额        环比
日期                              
2024-01-01  100.000000       NaN
2024-01-02  125.000000  0.500000
2024-01-03  126.666667  0.183333
2024-02-01  145.000000  0.301709
60
DatetimeIndex(['2024-01-01', '2024-01-03', '2024-01-05', '2024-01-07',
               '2024-01-09'],
              dtype='datetime64[us]', freq='2D')
2023-12-31 16:00:00+00:00
